# Objective
Here, we'll implement our new interpolation function and apply it to our entire cohort. We'll also establish a 'max-days' threshold for our measurements. 

In [ ]:
# load pacakges
suppressPackageStartupMessages(library(maplet))
suppressPackageStartupMessages(require(here))

In [ ]:
# load interpolation and locf function 
source(here('helpers/locf_exact_impute.r'))
source(here('helpers/interp_locf.r'))

In [ ]:
# load metadata (with KPRB enrollment date)
meta <- read.csv(here('data/meta.csv'), row.names = 1) %>% 
    # ensure KPRB_SAMPLE_DATE is a date
    mutate(KPRB_SAMPLE_DATE = as.Date(KPRB_SAMPLE_DATE)) %>% 
    select(StudyID, KPRB_SAMPLE_DATE, age_at_sample, sex)

# CLIN Variables

In [ ]:
# read bmi
bmi <- read.csv(here('data/bmi.csv'), row.names = 1) %>% 
    right_join(meta, by = "StudyID") # join with meta for enrollment date
head(bmi)

In [ ]:
# estimate enrollment date bmi
bmi_df <- estimate_index_value(
  df                    = bmi,
  value_col             = "BMI",
  index_date_col        = "KPRB_SAMPLE_DATE",
  measurement_date_col  = "MEASUREMENT_DATE",
  proximal_days         = 90, # non-interp window of proximal values 
  interp_before_days    = 365,
  interp_after_days     = 365,
  locf_days             = 730,
  output_col            = "BMI_interp"
)

In [ ]:
# load sbp data
sbp <- read.csv(here('data/sbp.csv'), row.names = 1) %>% 
    right_join(meta, by = "StudyID") # join with meta for enrollment date

In [ ]:
# estimate enrollment date sbp 
sbp_df <- estimate_index_value(
  df                    = sbp,
  value_col             = "Systolic",
  index_date_col        = "KPRB_SAMPLE_DATE",
  measurement_date_col  = "MEASUREMENT_DATE",
  proximal_days         = 90,
  interp_before_days    = 365,
  interp_after_days     = 365,
  locf_days             = 730,

  output_col            = "SBP_interp"
)

# Labs

In [ ]:
# median same-day labs, join KPRB sample date, then LOCF-impute.
# People in meta with no lab row are kept, so the missingness block can
# zero-gate them instead of introducing NAs at a later join.
prep_locf_impute <- function(labs_df,
                             meta_df = meta,
                             value_col = "Result_C",
                             date_col = "RESULT_DATE",
                             index_date_col = "KPRB_SAMPLE_DATE",
                             exact_window = 0,
                             locf_window = 730) {

  cohort <- meta_df %>%
    distinct(StudyID, .data[[index_date_col]])

  labs <- labs_df %>%
    group_by(StudyID, .data[[date_col]]) %>%
    summarise(
      !!value_col := median(.data[[value_col]], na.rm = TRUE),
      .groups = "drop"
    ) %>%
    ungroup() %>%
    inner_join(cohort, by = "StudyID") %>%
    rename(MEASUREMENT_DATE = !!rlang::sym(date_col)) %>%
    mutate(MEASUREMENT_DATE = as.Date(MEASUREMENT_DATE))

  no_measurement <- cohort %>%
    anti_join(labs, by = "StudyID") %>%
    mutate(
      MEASUREMENT_DATE = as.Date(NA),
      !!value_col := NA_real_
    )

  locf_exact_impute(
    bind_rows(labs, no_measurement),
    value_col      = value_col,
    index_date_col = index_date_col,
    date_col       = "MEASUREMENT_DATE",
    exact_window   = exact_window,
    locf_window    = locf_window
  )
}

In [ ]:
# load consolidated labs
all_clin_labs <- read.csv(here('data/labs.csv')) %>%
  select(StudyID, Test_Type, Result_C, RESULT_DATE) %>% # select lab type, result, and date
  mutate(
    Result_C = as.numeric(Result_C),
    RESULT_DATE = as.Date(RESULT_DATE)
  )

# subset to individual lab files
a1c_labs <- all_clin_labs %>% filter(Test_Type == 'HGBA1C')
creat_labs <- all_clin_labs %>% filter(Test_Type == 'CREATININE')
alt_labs <- all_clin_labs %>% filter(Test_Type == 'ALT')
ldl_labs <- all_clin_labs %>% filter(Test_Type == 'LDL_CLC_F' | Test_Type == 'LDL_CLC_NS' | Test_Type == 'LDL_DIRECT' | Test_Type == 'LDL_NS')
ast_labs <- all_clin_labs %>% filter(Test_Type == 'AST')
platelets_labs <- all_clin_labs %>% filter(Test_Type == 'PLATELETS')
hdl_labs <- all_clin_labs %>% filter(Test_Type == 'HDL')
total_cholesterol_labs <- all_clin_labs %>% filter(Test_Type == 'TOT_CHOLES')
albumin_labs <- all_clin_labs %>% filter(Test_Type == 'ALBUMIN')
triglyc_labs <- all_clin_labs %>% filter(Test_Type == 'TRIGL_F' | Test_Type == 'TRIGL_NS')
hgb_labs <- all_clin_labs %>% filter(Test_Type == 'HGB')

In [ ]:
# Zero-gate the imputed value and lookback delay into a three-parameter block.
# If the imputed result is missing, M = 0 and both the value and the delay are 0.
# If it is present, M = 1 and the observed value and delay are kept.
name_block_cols <- function(df, obs_col, time_col, miss_col) {
  observed <- !is.na(df$imputed_Result_C)
  df %>%
    dplyr::transmute(
      StudyID,
      !!miss_col := dplyr::if_else(observed, 1L, 0L),
      !!obs_col  := dplyr::if_else(observed, as.numeric(imputed_Result_C), 0),
      !!time_col := dplyr::if_else(observed, as.numeric(days_before), 0)
    )
}

# estimate individual labs
a1c_df <- prep_locf_impute(a1c_labs, meta_df = meta) %>%
  name_block_cols("A1C_obs", "T_a1c", "M_a1c")

ldl_df <- prep_locf_impute(ldl_labs, meta_df = meta) %>%
  name_block_cols("LDL_obs", "T_ldl", "M_ldl")

creat_df <- prep_locf_impute(creat_labs, meta_df = meta) %>%
  name_block_cols("CREAT_obs", "T_creat", "M_creat")

alt_df <- prep_locf_impute(alt_labs, meta_df = meta) %>%
  name_block_cols("ALT_obs", "T_alt", "M_alt")

ast_df <- prep_locf_impute(ast_labs, meta_df = meta) %>%
  name_block_cols("AST_obs", "T_ast", "M_ast")

platelets_df <- prep_locf_impute(platelets_labs, meta_df = meta) %>%
  name_block_cols("PLATELETS_obs", "T_platelets", "M_platelets")

hdl_df <- prep_locf_impute(hdl_labs, meta_df = meta) %>%
  name_block_cols("HDL_CHOLES_obs", "T_hdl_choles", "M_hdl_choles")

totalchol_df <- prep_locf_impute(total_cholesterol_labs, meta_df = meta) %>%
  name_block_cols("TOT_CHOLES_obs", "T_tot_choles", "M_tot_choles")

albumin_df <- prep_locf_impute(albumin_labs, meta_df = meta) %>%
  name_block_cols("ALBUMIN_obs", "T_albumin", "M_albumin")

triglyc_df <- prep_locf_impute(triglyc_labs, meta_df = meta) %>%
  name_block_cols("TRIGLYCERIDES_obs", "T_triglycerides", "M_triglycerides")

hgb_df <- prep_locf_impute(hgb_labs, meta_df = meta) %>%
  name_block_cols("HGB_obs", "T_hgb", "M_hgb")

# eGFR

In [ ]:
# Compute eGFR with the 2021 CKD-EPI creatinine equation.
egfr_df <- meta %>%
    left_join(creat_df, by = "StudyID") %>%
    mutate(
        sex_num = as.integer(as.character(sex)),
        kappa = ifelse(sex_num == 0, 0.7, 0.9),
        alpha = ifelse(sex_num == 0, -0.241, -0.302),
        egfr = if_else(
            M_creat == 1L, # only applies if creatinine was observed
            142 *
                pmin(CREAT_obs / kappa, 1)^alpha *
                pmax(CREAT_obs / kappa, 1)^(-1.200) *
                0.9938^age_at_sample *
                ifelse(sex_num == 0, 1.012, 1),
            NA_real_
        ),
        # apply zero-gated logic 
        M_egfr = if_else(!is.na(egfr), 1L, 0L),
        EGFR_obs = if_else(!is.na(egfr), as.numeric(egfr), 0),
        T_egfr = if_else(M_egfr == 1L, as.numeric(T_creat), 0)
    ) %>%
    select(-sex_num, -kappa, -alpha)

# FIB-4

In [ ]:
# FIB-4 at the index date from the most recent pre-index ALT, AST, and platelets.
# The score is computed only when all three labs were observed. The delay is the stalest of the three.
fib4_df <- meta %>%
    left_join(alt_df, by = "StudyID") %>%
    left_join(ast_df, by = "StudyID") %>%
    left_join(platelets_df, by = "StudyID") %>%
    mutate(
        components_observed = M_alt == 1L & M_ast == 1L & M_platelets == 1L,
        fib4 = if_else(
            components_observed,
            (age_at_sample * AST_obs) / (PLATELETS_obs * sqrt(ALT_obs)),
            NA_real_
        ),
        fib4_days_before = if_else(
            components_observed,
            pmax(T_alt, T_ast, T_platelets),
            NA_real_
        ),
        M_fib4 = if_else(!is.na(fib4), 1L, 0L),
        FIB4_obs = if_else(!is.na(fib4), as.numeric(fib4), 0),
        T_fib4 = if_else(M_fib4 == 1L, as.numeric(fib4_days_before), 0)
    ) %>%
    select(-components_observed)

# PREVENT

## BP and Statin Treatment

In [ ]:
source(here::here("helpers", "derive_baseline_bp_statin_status.r"))

# load rx
rx <- read.csv(here('data/rx.csv'))
bp_statin_treatment_df <- derive_baseline_bp_statin_status(meta, rx) # compute baseline bp and statin treatment

## Smoking

In [ ]:
# load smooking
smoking <- read.csv(here('data/smoking.csv'))

# code smoking for prevent-compatibility
smoking_prevent <- smoking %>% 
    ### generate model for smoking 
    mutate(
    # presence indicator: did the participant answer the smoking_status question?
    M_smoke = if_else(smoking_status != "Missing" & !is.na(smoking_status), 1L, 0L),

    # status dummies, reference = Never, zero-gated when not answered
    smk_current = if_else(M_smoke == 1L & smoking_status == "Current", 1L, 0L),
    smk_former  = if_else(M_smoke == 1L & smoking_status == "Former",  1L, 0L)
    ) %>% 
    select(StudyID, M_smoke, smk_current, smk_former)

# CHARGE-AF
CHARGE-AF uniquely requires height, weight, and DBP covariates. While this creates some redudnancy with CLIN (e.g. height+weight are redundant when BMI is in model), to ensure full compliance with the CHARGE-AF model, we define and include these variables nonetheless. This is justified given the use of regularied models which can handle statstical collinearity among SOC predictors. We define remaining CHARGE-AF ingredients here. Similar to our rationale for CLIN predictors – whereby BMI and SBP are routinately measured and have little overall missingness, we'll similarly estimate DBP, Height/Weight. 

## Physical Measurements

In [ ]:
# height 
height <- read.csv(here('data/height.csv'), row.names = 1) %>% 
    right_join(meta, by = "StudyID") # join with meta for enrollment date

height_df <- estimate_index_value(
  df                    = height,
  value_col             = "height_m",
  index_date_col        = "KPRB_SAMPLE_DATE",
  measurement_date_col  = "MEASUREMENT_DATE",
  output_col            = "height_interp"
)

# weight
weight <- read.csv(here('data/weight.csv'), row.names = 1) %>% 
    right_join(meta, by = "StudyID") # join with meta for enrollment date

weight_df <- estimate_index_value(
  df                    = weight,
  value_col             = "weight_kg",
  index_date_col        = "KPRB_SAMPLE_DATE",
  measurement_date_col  = "MEASUREMENT_DATE",
  output_col            = "weight_interp"
)

# dbp
dbp <- read.csv(here('data/dbp.csv'), row.names = 1) %>% 
    right_join(meta, by = "StudyID") # join with meta for enrollment date

dbp_df <- estimate_index_value(
  df                    = dbp,
  value_col             = "Diastolic",
  index_date_col        = "KPRB_SAMPLE_DATE",
  measurement_date_col  = "MEASUREMENT_DATE",
  output_col            = "DBP_interp"
)

## Smoking

In [ ]:
# code smoking for chargeaf compatability 
smoking_charge_af <- smoking %>%
    mutate(
        M_smoke_af = if_else(smoking_status == "Missing", 0L, 1L), # zero gated when not answered
        smoking_current_af = if_else(
            M_smoke_af == 1L & smoking_status == "Current", 1L, 0L # binarize current smokers
        )
    ) %>% 
    select(StudyID, M_smoke_af, smoking_current_af)

## Race 
Similarly, CHARGE-AF uniquely includes a 'race' variable that binarizes self-reported 'race' as 1 if 'White', else 0. We'll similarly incorporate this coding schema here.

In [ ]:
# load race 
charge_af_race <- read.csv(here('data/meta.csv'), row.names = 1) %>% 
    select(StudyID, vdw_race_hisp) %>% 
    mutate(race = case_when(
        vdw_race_hisp == 'White' ~ 1,
        TRUE ~ 0
    )) %>% 
    select(StudyID, race)

In [ ]:
charge_af_race %>% dplyr::count(race)

# SOC+ Predictors

## A1C Variability

In [ ]:
# load A1C variability helper
source(here('helpers', 'compute_a1c_variability.r'))

# compute per-patient pre-enrollment A1C variability (SD / CV), no imputation
# - backward-only 2-year window (no leakage), same-day draws collapsed to median
# - >= 2 contributing points required; M/obs/T zero-gated encoding returned
a1c_var <- compute_a1c_variability(
    labs_df           = a1c_labs,
    meta_df           = meta,
    id_col            = 'StudyID',
    value_col         = 'Result_C',
    date_col          = 'RESULT_DATE',
    index_date_col    = 'KPRB_SAMPLE_DATE',
    lookback_days     = 730,    # 2-year max lookback
    min_points        = 2,      # >= 2 for variability (3+ more stable)
    include_index_day = TRUE,
    value_range       = c(3, 20)  # drop sentinel/implausible A1C (raw feed has -1e100)
)

## eGFR Slope

In [ ]:
# load eGFR slope helper
source(here('helpers', 'compute_egfr_slope.r'))

# compute per-patient pre-enrollment eGFR slope (renal decline), no imputation
# - creatinine -> eGFR FIRST (2021 CKD-EPI, age at each RESULT_DATE), then slope on eGFR
# - STRICT pre-enrollment only (meas_date < enrollment), 5-year max lookback
# - same-day draws collapsed to median; >= 3 points required for a stable slope
# - sign: negative slope = declining eGFR (forward-time axis)
egfr_slope <- compute_egfr_slope(
    labs_df        = creat_labs,
    meta_df        = meta,
    id_col         = 'StudyID',
    value_col      = 'Result_C',
    date_col       = 'RESULT_DATE',
    index_date_col = 'KPRB_SAMPLE_DATE',
    age_col        = 'age_at_sample',
    sex_col        = 'sex',
    lookback_days  = 1826,        # 5-year max lookback for trajectory
    min_points     = 3,           # >= 3 for a stable slope (2 is the floor)
    creat_range    = c(0.1, 20)   # drop implausible creatinine (raw feed has 0 and 333)
)

## NAFLD Fibrosis Score

In [ ]:
# load diabetes status 
diab_prev <- read.csv(here('data/meta.csv'), row.names = 1) %>% 
    select(StudyID, diabetes_at_baseline)

In [ ]:
# NAFLD fibrosis score at the index date.
# Labs are already zero-gated, so a missing ALT is 0 and AST/ALT is Inf.
# Compute the score only when ALT, AST, platelets, albumin, BMI, age, and
# diabetes status are all available.
nfs_df <- meta %>%
    inner_join(diab_prev, by = "StudyID") %>%
    select(StudyID, age_at_sample, diabetes_at_baseline) %>%
    left_join(bmi_df %>% select(StudyID, BMI_interp), by = "StudyID") %>%
    left_join(alt_df, by = "StudyID") %>%
    left_join(ast_df, by = "StudyID") %>%
    left_join(platelets_df, by = "StudyID") %>%
    left_join(albumin_df, by = "StudyID") %>%
    mutate(
        labs_observed = M_alt == 1L & M_ast == 1L & M_platelets == 1L & M_albumin == 1L,
        score_observed = labs_observed &
            !is.na(age_at_sample) & !is.na(BMI_interp) & !is.na(diabetes_at_baseline),
        nfs = if_else(
            score_observed,
            -1.675 +
                0.037 * age_at_sample +
                0.094 * BMI_interp +
                1.13 * diabetes_at_baseline +
                0.99 * (AST_obs / ALT_obs) -
                0.013 * PLATELETS_obs -
                0.66 * ALBUMIN_obs,
            NA_real_
        ),
        nfs_days_before = if_else(
            labs_observed,
            pmax(T_alt, T_ast, T_platelets, T_albumin),
            NA_real_
        ),
        M_nfs = if_else(!is.na(nfs), 1L, 0L),
        NFS_obs = if_else(!is.na(nfs), as.numeric(nfs), 0),
        T_nfs = if_else(M_nfs == 1L, as.numeric(nfs_days_before), 0)
    ) %>%
    select(-labs_observed, -score_observed)

summary(nfs_df$nfs)

# Consolidate

In [ ]:
# load full metadata and incorporate clinical parameters
meta <- read.csv(here('data', 'meta.csv'), row.names = 1) %>% 
  mutate(KPRB_SAMPLE_DATE = as.Date(KPRB_SAMPLE_DATE)) # ensure representation as date

In [ ]:
# Join one StudyID-keyed frame onto meta, keeping only columns that are
# not already present. Shared names (KPRB_SAMPLE_DATE, age, method, and the
# lab ingredients carried inside egfr/fib4/nfs) are not selected, so joins
# cannot create .x/.y suffixes.
add_by_id <- function(base, df, cols) {
  overlap <- intersect(names(base), cols)
  if (length(overlap) > 0) {
    stop("Column already on meta: ", paste(overlap, collapse = ", "), call. = FALSE)
  }
  n0 <- nrow(base)
  out <- base %>%
    left_join(df %>% select(StudyID, all_of(cols)), by = "StudyID")
  if (nrow(out) != n0) {
    stop("Join changed nrow while adding: ", paste(cols, collapse = ", "), call. = FALSE)
  }
  out
}

# join all clinical parameters
meta <- meta %>%
  add_by_id(bmi_df, "BMI_interp") %>%
  add_by_id(sbp_df, "SBP_interp") %>%
  add_by_id(a1c_df, c("M_a1c", "A1C_obs", "T_a1c")) %>%
  add_by_id(a1c_var, c("M_a1c_var", "A1C_cv_obs", "T_a1c_var")) %>%
  add_by_id(ldl_df, c("M_ldl", "LDL_obs", "T_ldl")) %>%
  add_by_id(hdl_df, c("M_hdl_choles", "HDL_CHOLES_obs", "T_hdl_choles")) %>%
  add_by_id(totalchol_df, c("M_tot_choles", "TOT_CHOLES_obs", "T_tot_choles")) %>%
  add_by_id(triglyc_df, c("M_triglycerides", "TRIGLYCERIDES_obs", "T_triglycerides")) %>%
  add_by_id(albumin_df, c("M_albumin", "ALBUMIN_obs", "T_albumin")) %>%
  add_by_id(hgb_df, c("M_hgb", "HGB_obs", "T_hgb")) %>%
  add_by_id(egfr_df, c("M_egfr", "EGFR_obs", "T_egfr")) %>%
  add_by_id(egfr_slope, c("M_egfr_slope", "EGFR_slope_obs", "T_egfr_slope")) %>%
  add_by_id(fib4_df, c("M_fib4", "FIB4_obs", "T_fib4")) %>%
  add_by_id(nfs_df, c("M_nfs", "NFS_obs", "T_nfs")) %>%
  add_by_id(bp_statin_treatment_df, c("bp_treatment", "statin_treatment")) %>%
  add_by_id(smoking_prevent, c("M_smoke", "smk_current", "smk_former")) %>%
  add_by_id(smoking_charge_af, c("M_smoke_af", "smoking_current_af")) %>%
  add_by_id(height_df, "height_interp") %>%
  add_by_id(weight_df, "weight_interp") %>%
  add_by_id(dbp_df, "DBP_interp") %>%
  add_by_id(charge_af_race, "race")

# save 
write.csv(meta, here('data/meta_clin.csv'))

In [ ]:
# Baseline table for comparison with the manuscript.
fmt_med_iqr <- function(x) {
  q <- stats::quantile(x, c(0.25, 0.5, 0.75), na.rm = TRUE)
  sprintf("%.1f (%.1f-%.1f)", q[2], q[1], q[3])
}
fmt_n <- function(n, N) sprintf("%s (%.1f%%)", format(n, big.mark = ","), 100 * n / N)

N <- nrow(meta)
med <- function(x, measured) fmt_med_iqr(x[measured])
n_meas <- function(measured) fmt_n(sum(measured, na.rm = TRUE), N)
n_flag <- function(x) fmt_n(sum(x == 1, na.rm = TRUE), N)
index_years <- paste(range(format(meta$KPRB_SAMPLE_DATE, "%Y"), na.rm = TRUE), collapse = "-")

# summarize statistics
baseline_table <- tibble::tibble(
  Measurement = c(
    "BMI, median (IQR)", "BMI measured, n (%)",
    "Systolic blood pressure, median (IQR)", "Systolic blood pressure measured, n (%)",
    "Diastolic blood pressure, median (IQR)", "Diastolic blood pressure measured, n (%)",
    "HbA1c, median (IQR)", "HbA1c measured, n (%)",
    "LDL cholesterol, median (IQR)", "LDL cholesterol measured, n (%)",
    "HDL cholesterol, median (IQR)", "HDL cholesterol measured, n (%)",
    "Total cholesterol, median (IQR)", "Total cholesterol measured, n (%)",
    "eGFR, median (IQR)", "eGFR measured, n (%)",
    "FIB-4, median (IQR)", "FIB-4 measured, n (%)",
    "Blood pressure treatment, n (%)",
    "Statin treatment, n (%)",
    "Prevalent diabetes, n (%)",
    "Prevalent heart failure, n (%)",
    "Prevalent myocardial infarction, n (%)",
    "Current smoker, n (%)",
    "Former smoker, n (%)",
    "Index/baseline period",
    "Biospecimen"
  ),
  Value = c(
    med(meta$BMI_interp, !is.na(meta$BMI_interp)), n_meas(!is.na(meta$BMI_interp)),
    med(meta$SBP_interp, !is.na(meta$SBP_interp)), n_meas(!is.na(meta$SBP_interp)),
    med(meta$DBP_interp, !is.na(meta$DBP_interp)), n_meas(!is.na(meta$DBP_interp)),
    med(meta$A1C_obs, meta$M_a1c == 1), n_meas(meta$M_a1c == 1),
    med(meta$LDL_obs, meta$M_ldl == 1), n_meas(meta$M_ldl == 1),
    med(meta$HDL_CHOLES_obs, meta$M_hdl_choles == 1), n_meas(meta$M_hdl_choles == 1),
    med(meta$TOT_CHOLES_obs, meta$M_tot_choles == 1), n_meas(meta$M_tot_choles == 1),
    med(meta$EGFR_obs, meta$M_egfr == 1), n_meas(meta$M_egfr == 1),
    med(meta$FIB4_obs, meta$M_fib4 == 1), n_meas(meta$M_fib4 == 1),
    n_flag(meta$bp_treatment),
    n_flag(meta$statin_treatment),
    n_flag(meta$diabetes_at_baseline),
    n_flag(meta$hf_at_baseline),
    n_flag(meta$mi_at_baseline),
    n_flag(meta$smk_current),
    n_flag(meta$smk_former),
    index_years,
    "Serum"
  )
)

# save 
write.csv(baseline_table, here('outputs/tables/LABS_MEDS.csv'))

# Longitudinal Variables for Patient Plots
Since we'll later visualize our SOC and NMR-enhanced risk scores alongside longitudinal EHR data. We'll prepare our longitudinal biomarkers for this analysis here. 

In [ ]:
# a1c (obtained above)
write.csv(a1c_labs, here('data/a1c_history.csv'))

In [ ]:
# Longitudinal eGFR and FIB-4 for patient plots.
# These are full lab histories, not the index-date scores joined onto meta above.
# creat_labs / ast_labs / alt_labs / platelets_labs already carry StudyID,
# Result_C, and RESULT_DATE. meta is reloaded below from meta.csv (no labs)
# for StudyID, KPRB_SAMPLE_DATE, age_at_sample, and sex.

# load longitudinal eGFR and FIB-4 history helpers
source(here('helpers', 'compute_egfr_history.r'))
source(here('helpers', 'compute_fib4_history.r'))

# load clean meta data (no labs) 
meta <- read.csv(here('data/meta.csv'), row.names = 1)

# convert every creatinine result to eGFR (2021 CKD-EPI, race-free)
# - no lookback window and no pre-/post-enrollment restriction
# - age_at_result is age_at_sample shifted to RESULT_DATE
# - collapse_same_day = FALSE keeps the full observed lab history
egfr_history <- compute_egfr_history(
    labs_df           = creat_labs,
    meta_df           = meta,
    id_col            = 'StudyID',
    value_col         = 'Result_C',
    date_col          = 'RESULT_DATE',
    index_date_col    = 'KPRB_SAMPLE_DATE',
    age_col           = 'age_at_sample',
    sex_col           = 'sex',
    test_type_col     = 'Test_Type',
    creat_range       = c(0.1, 20),
    collapse_same_day = FALSE
)

# save for patient-centric plotting workflows
write.csv(egfr_history, here('data/egfr_history.csv'), row.names = FALSE)

# build a full FIB-4 history for trend plotting (not for modeling)
# - anchors on AST; nearest ALT and platelets within +/- window_days
# - a point is kept only when all three components fall in the window
# - same-day draws are median-collapsed inside the helper
# - fib4_date is the median of the three contributing dates
fib4_history <- compute_fib4_history(
    ast_df            = ast_labs,
    alt_df            = alt_labs,
    platelet_df       = platelets_labs,
    meta_df           = meta,
    id_col            = 'StudyID',
    value_col         = 'Result_C',
    date_col          = 'RESULT_DATE',
    index_date_col    = 'KPRB_SAMPLE_DATE',
    age_col           = 'age_at_sample',
    window_days       = 180,
    ast_range         = c(1, 2000),
    alt_range         = c(1, 2000),
    platelet_range    = c(1, 2000),
    collapse_same_day = TRUE
)

# save for patient-centric plotting workflows
write.csv(fib4_history, here('data/fib4_history.csv'), row.names = FALSE)